In [ ]:
# 1. Remove the standard MediaPipe that is clashing
!pip uninstall -y mediapipe protobuf

# 2. Install the modern versions that support NumPy 2.0+ and Protobuf 5.x
# We use the '--no-cache-dir' to ensure we don't grab a broken local copy
!pip install --no-cache-dir mediapipe==0.10.14
!pip install --no-cache-dir protobuf==5.29.5


# ASL Word Training (WLASL + NSLT split)

**Inputs**
- `WLASL_v0.3.json` plus a folder of videos named `{video_id}.mp4`
- One NSLT split file, e.g. `nslt_100.json`, to choose which clips/classes enter training
- Optional `asl_word_vocabulary.csv` in **work dir**; if missing, it is generated from the WLASL JSON

**How to run**

1. Run **cell 1** (MediaPipe protobuf fix), then **restart the notebook session**
2. In **Global Config**, set **`KAGGLE_DATA`** (on Kaggle) or **`PROJECT_ROOT`** / **`WORK_DIR`** (local) so paths resolve
3. Set **`NSLT_JSON`** for your split (`nslt_100` … `nslt_2000`)
4. Run all remaining cells top to bottom

**Rough Kaggle wall-clock (GPU notebook)**
- **Feature extraction** (MediaPipe FaceMesh + hands, no cache): often **hours** — usually the slowest step — scales with clip count and is faster with **`FRAME_SKIP`** and smaller **`RESIZE_WIDTH`**.
- **After** `_sequences.npz` exists and **`USE_CACHE_IF_EXISTS`** is on, extraction is skipped.
- **LSTM fitting** (~60 epochs max with early stopping): commonly **from ~15 min to a few hours** depending on GPU (e.g. P100/T4/L4), class count (`nslt_*`), and sample count.
- **`wlasl-2000-resized`** in **Cell 2** is only correct if **your added dataset folder** matches that layout; rename **`KAGGLE_DATA`** if not.


## Cell 1: Imports

_Three sub-cells: stdlib → heavy libs → sklearn/keras_


In [ ]:
import os
import json
from pathlib import Path
from collections import Counter
print("[Cell 1a/3] stdlib imports loaded")


In [ ]:
import cv2
import numpy as np
import pandas as pd
import tensorflow as tf
from tqdm import tqdm
import mediapipe as mp

from IPython.display import display
print("[Cell 1b/3] cv2 / numpy / pandas / TF / MediaPipe imported")


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Bidirectional, Dense, Dropout,
    BatchNormalization, TimeDistributed, Multiply,
    Activation, Lambda,
)
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical

print("✅ Imports loaded")
print("TensorFlow:", tf.__version__)


## Cell 2: Global Config

_Two sub-cells: path/language setup → auto-versioning_


In [ ]:
# =========================
# CELL 2: GLOBAL CONFIG — MANUAL PATHS
# =========================

# ── 1. Choose language ────────────────────────────────────────────────────
LANGUAGE = "asl"  # only "asl" supported here

# ── 2. Kaggle detection ───────────────────────────────────────────────────
IS_KAGGLE = Path('/kaggle/input').exists()
print('Running on Kaggle:', IS_KAGGLE)

# ── 3. Set paths manually ─────────────────────────────────────────────────
if IS_KAGGLE:
    # ── Kaggle: dataset = "WLASL-2000 Resized", folder = wlasl-complete ─────
    # Layout (as added via Add Data):
    #   /kaggle/input/wlasl-2000-resized/wlasl-complete/
    #     ├── videos/           {video_id}.mp4
    #     ├── WLASL_v0.3.json
    #     ├── wlasl_class_list.txt   (class_idx → gloss, tab-separated)
    #     ├── nslt_100.json  / nslt_300 / nslt_1000 / nslt_2000
    #     └── missing.txt
    KAGGLE_DATA      = Path('/kaggle/input/datasets/sttaseen/wlasl2000-resized/wlasl-complete')
    WLASL_JSON       = KAGGLE_DATA / '/kaggle/input/datasets/sttaseen/wlasl2000-resized/wlasl-complete/WLASL_v0.3.json'
    VIDEOS_DIR       = KAGGLE_DATA / '/kaggle/input/datasets/sttaseen/wlasl2000-resized/wlasl-complete/videos'
    WLASL_CLASS_LIST = KAGGLE_DATA / '/kaggle/input/datasets/sttaseen/wlasl2000-resized/wlasl-complete/wlasl_class_list.txt'
    WORK_DIR         = Path('/kaggle/working') / f'{LANGUAGE}_word_training'
    VOCAB_CSV        = WORK_DIR / 'asl_word_vocabulary.csv'
    # Change nslt_100 → nslt_300 / nslt_1000 / nslt_2000 to train on more classes
    NSLT_JSON        = KAGGLE_DATA / '/kaggle/input/datasets/sttaseen/wlasl2000-resized/wlasl-complete/nslt_100.json'

else:
    # ── Local ─────────────────────────────────────────────────────────────
    PROJECT_ROOT     = Path(r'M:/Term 10/Grad')
    WORK_DIR         = PROJECT_ROOT / 'SLR Main/Words/ASL Word (English)'
    WLASL_JSON       = WORK_DIR / 'WLASL_v0.3.json'
    NSLT_JSON        = WORK_DIR / 'nslt_100.json'
    WLASL_CLASS_LIST = WORK_DIR / 'wlasl_class_list.txt'
    VOCAB_CSV        = WORK_DIR / 'asl_word_vocabulary.csv'
    VIDEOS_DIR       = PROJECT_ROOT / 'Words dataset/Words Datasets/WLASL_videos'

# ── Extraction speed settings ────────────────────────────────────────────
FRAME_SKIP   = 4    # process every Nth frame (1=all, 2=half, 4=quarter)
RESIZE_WIDTH = 224  # resize frame width before MediaPipe (0 = no resize)

# ── 4. Validate paths ─────────────────────────────────────────────────────
for _p, _lbl in [
    (WLASL_JSON,       'WLASL_JSON'),
    (WLASL_CLASS_LIST, 'WLASL_CLASS_LIST'),
    (NSLT_JSON,        'NSLT_JSON'),
    (VIDEOS_DIR,       'VIDEOS_DIR'),
]:
    if not Path(_p).exists():
        print(f'⚠️  Not found: {_lbl} = {_p}')
    else:
        print(f'✅ Found : {_lbl}')

WORK_DIR.mkdir(parents=True, exist_ok=True)

# ── 5. Language config dict ────────────────────────────────────────────────
CFG = {
    'asl': {
        'name'              : 'ASL (English)',
        'work_dir'          : WORK_DIR,
        'vocab_csv'         : VOCAB_CSV,
        'dataset_type'      : 'wlasl',
        'wlasl_json'        : WLASL_JSON,
        'wlasl_class_list'  : WLASL_CLASS_LIST,
        'videos_dir'        : VIDEOS_DIR,
        'sequence_len'      : 30,
        'features_per_frame': 1530,  # 1404 (FaceMesh) + 63 (LH) + 63 (RH)
        'nslt_json'         : NSLT_JSON,
    },
}

C = CFG[LANGUAGE]

print('=' * 60)
print(f'[Cell 2] Config ready  |  language={LANGUAGE}  |  kaggle={IS_KAGGLE}')
if IS_KAGGLE: print(f'  KAGGLE_DATA: {KAGGLE_DATA}')
print(f'  NSLT_JSON : {NSLT_JSON.name}')
print(f'  WORK_DIR  : {WORK_DIR}')
print('=' * 60)



In [ ]:
# ==========================================
# 🌟 AUTO-NUMBERING LOGIC FOR MODELS 🌟
# Checks folder for v1, v2, v3 and takes the next available number
# ==========================================
version = 1
while True:
    test_path = WORK_DIR / f"{LANGUAGE}_word_lstm_model_best_v{version}.h5"
    if not test_path.exists():
        break
    version += 1

# Only version the models (saves them as v1, v2, v3...)
MODEL_BEST = WORK_DIR / f"{LANGUAGE}_word_lstm_model_best_v{version}.h5"
MODEL_FINAL = WORK_DIR / f"{LANGUAGE}_word_lstm_model_final_v{version}.h5"

# Keep the cache files static so you don't have to wait 2 hours for re-extraction!
CACHE_NPZ = WORK_DIR / f"{LANGUAGE}_word_sequences.npz"
CLASSES_CSV = WORK_DIR / f"{LANGUAGE}_word_classes.csv"
SCALER_STATS = WORK_DIR / f"{LANGUAGE}_scaler_stats.npz"

SEQUENCE_LENGTH = int(C["sequence_len"])
FEATURES_PER_FRAME = int(C["features_per_frame"])

print("=" * 60)
print(f"🌟 RUN CONFIG — TRAINING VERSION v{version} 🌟")
print("=" * 60)
print(f"Output Best  : {MODEL_BEST.name}")
print(f"Output Final : {MODEL_FINAL.name}")
print("LANGUAGE:", LANGUAGE, "|", C["name"])


## Cell 3: GPU Setup


In [ ]:
print("=" * 60)
print("GPU SETUP")
print("=" * 60)
gpus = tf.config.list_physical_devices("GPU")
print("Detected GPUs:", gpus)
if gpus:
    try:
        for g in gpus:
            tf.config.experimental.set_memory_growth(g, True)
        print("✅ Memory growth enabled")
    except RuntimeError as e:
        print("⚠️ GPU setup warning:", e)
else:
    print("⚠️ No GPU detected; CPU mode.")

tf.random.set_seed(42)
np.random.seed(42)


## Cell 4: Load Language Vocab


In [ ]:
# ── Vocab CSV: load directly or auto-generate from WLASL_v0.3.json ──
vocab_path = Path(C['vocab_csv'])

if not vocab_path.exists():
    print(f'[Cell 4] vocab CSV not found at {vocab_path}')
    print('[Cell 4] Auto-generating vocab ...')

    _cl_path = C.get('wlasl_class_list')
    if _cl_path and Path(_cl_path).exists():
        # Primary source: wlasl_class_list.txt  (class_idx TAB gloss)
        # Index here == WLASL JSON array index == nslt action[0] class index
        _rows = []
        with open(_cl_path, 'r', encoding='utf-8') as _f:
            for _line in _f:
                _parts = _line.strip().split('\t', 1)
                if len(_parts) == 2:
                    _idx, _gloss = int(_parts[0]), _parts[1].strip()
                    _rows.append({'word_id': _idx, 'label_name': _gloss, 'source_class_id': _idx})
        vocab = pd.DataFrame(_rows)
        vocab_path.parent.mkdir(parents=True, exist_ok=True)
        vocab.to_csv(vocab_path, index=False)
        print(f'[Cell 4] Generated vocab from wlasl_class_list.txt: {len(vocab)} classes -> {vocab_path}')

    else:
        # Fallback: WLASL_v0.3.json (gloss at array index == class_id)
        _wlasl_path = C['wlasl_json']
        if not Path(_wlasl_path).exists():
            raise FileNotFoundError(
                f'Cannot generate vocab: neither wlasl_class_list.txt nor WLASL_v0.3.json found.'
            )
        with open(_wlasl_path, 'r', encoding='utf-8') as _f:
            _wlasl_data = json.load(_f)
        _rows = []
        for _idx, _item in enumerate(_wlasl_data):
            _label = str(_item.get('gloss', '')).strip()
            if _label:
                _rows.append({'word_id': _idx, 'label_name': _label, 'source_class_id': _idx})
        vocab = pd.DataFrame(_rows)
        vocab_path.parent.mkdir(parents=True, exist_ok=True)
        vocab.to_csv(vocab_path, index=False)
        print(f'[Cell 4] Generated vocab from WLASL_v0.3.json: {len(vocab)} classes -> {vocab_path}')
else:
    vocab = pd.read_csv(vocab_path)
    print(f'[Cell 4] Loaded existing vocab: {vocab_path} ({len(vocab)} rows)')


In [ ]:
# ── Known column aliases from older vocab CSV formats ──────────────────
_LABEL_ALIASES    = ['english', 'gloss', 'word', 'label']
_CLASSID_ALIASES  = ['wlasl_class', 'class_id', 'gloss_id', 'id', 'source_id']

def _try_remap(df):
    """Rename known alternative column names to the canonical ones."""
    renames = {}
    cols = set(df.columns)
    if 'label_name' not in cols:
        for a in _LABEL_ALIASES:
            if a in cols:
                renames[a] = 'label_name'
                break
    if 'source_class_id' not in cols:
        for a in _CLASSID_ALIASES:
            if a in cols:
                renames[a] = 'source_class_id'
                break
    return df.rename(columns=renames) if renames else df

def _regen_vocab_from_wlasl(wlasl_json_path):
    """Rebuild vocab DataFrame from WLASL_v0.3.json."""
    if not Path(wlasl_json_path).exists():
        raise FileNotFoundError(f'Cannot regenerate vocab: WLASL JSON not found: {wlasl_json_path}')
    with open(wlasl_json_path, 'r', encoding='utf-8') as _f:
        _data = json.load(_f)
    _rows = []
    for _idx, _item in enumerate(_data):
        _label = str(_item.get('gloss', '')).strip()
        if not _label:
            continue
        _cid = _item.get('gloss_id', _item.get('class_id', _item.get('id', _idx)))
        try:
            _cid = int(_cid)
        except (TypeError, ValueError):
            _cid = _idx
        _rows.append({'word_id': _idx, 'label_name': _label, 'source_class_id': _cid})
    return pd.DataFrame(_rows)

# ── Attempt remap first; regenerate as fallback ─────────────────────────
vocab = _try_remap(vocab)
required_cols = {'label_name', 'source_class_id'}
still_missing = required_cols - set(vocab.columns)

if still_missing:
    print(f'⚠️  Vocab CSV columns {vocab.columns.tolist()} lack {still_missing}.')
    print('   Regenerating from WLASL_v0.3.json ...')
    vocab = _regen_vocab_from_wlasl(C['wlasl_json'])
    vocab.to_csv(vocab_path, index=False)
    print(f'   Saved regenerated vocab -> {vocab_path}')

vocab['label_name']     = vocab['label_name'].astype(str).str.strip()
vocab['source_class_id'] = vocab['source_class_id'].astype(int)
if 'word_id' in vocab.columns:
    vocab['word_id'] = vocab['word_id'].astype(int)

allowed_class_ids = set(vocab['source_class_id'].tolist())
classid_to_label  = dict(zip(vocab['source_class_id'], vocab['label_name']))

print(f'✅ Loaded vocab rows: {len(vocab)}')
print(f'✅ Allowed classes  : {len(allowed_class_ids)}')
display(vocab.head())


## Cell 5: MediaPipe Helpers

_Two sub-cells: keypoint extractor → sequence padder_


In [ ]:
# MediaPipe Tasks API — replaces the broken legacy Solutions API on Python 3.12 / Kaggle.
# The legacy mp.solutions.face_mesh fails with 'Failed to parse' / 'ValidatedGraphConfig'
# errors on newer Python builds because its bundled binary graph can't be deserialized.
# The Tasks API uses proper .task model files and works correctly.
import urllib.request, os
from mediapipe.tasks import python as _mp_tasks
from mediapipe.tasks.python import vision as _mp_vision
from mediapipe import Image as _MpImage, ImageFormat as _MpFmt

_FACE_PATH = '/tmp/face_landmarker.task'
_HAND_PATH = '/tmp/hand_landmarker.task'
_FACE_URL  = ('https://storage.googleapis.com/mediapipe-models/'
               'face_landmarker/face_landmarker/float16/1/face_landmarker.task')
_HAND_URL  = ('https://storage.googleapis.com/mediapipe-models/'
               'hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task')

for _p, _u in ((_FACE_PATH, _FACE_URL), (_HAND_PATH, _HAND_URL)):
    if not os.path.exists(_p):
        print(f'Downloading {os.path.basename(_p)} ...')
        urllib.request.urlretrieve(_u, _p)
        print('  done.')

_face_lm = _mp_vision.FaceLandmarker.create_from_options(
    _mp_vision.FaceLandmarkerOptions(
        base_options=_mp_tasks.BaseOptions(model_asset_path=_FACE_PATH),
        running_mode=_mp_vision.RunningMode.IMAGE,
        num_faces=1,
        output_face_blendshapes=False,
        output_facial_transformation_matrixes=False,
    )
)
_hand_lm = _mp_vision.HandLandmarker.create_from_options(
    _mp_vision.HandLandmarkerOptions(
        base_options=_mp_tasks.BaseOptions(model_asset_path=_HAND_PATH),
        running_mode=_mp_vision.RunningMode.IMAGE,
        num_hands=2,
    )
)

def extract_tier3_keypoints(frame_bgr, *_):
    """Extract 1530 features: FaceMesh(1404) + left_hand(63) + right_hand(63).
    *_ absorbs legacy face_obj / hands_obj args so old call-sites still work.
    """
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_img = _MpImage(image_format=_MpFmt.SRGB, data=frame_rgb)

    # 1. FaceMesh: up to 478 landmarks from Tasks API; take first 468 -> 1404 features
    face = np.zeros(1404, dtype=np.float32)
    face_res = _face_lm.detect(mp_img)
    if face_res.face_landmarks:
        lms = face_res.face_landmarks[0][:468]
        face = np.array([[lm.x, lm.y, lm.z] for lm in lms], dtype=np.float32).flatten()[:1404]

    # 2. Hands: 21 x 3 = 63 per hand
    lh = np.zeros(63, dtype=np.float32)
    rh = np.zeros(63, dtype=np.float32)
    has_lh = has_rh = False
    hand_res = _hand_lm.detect(mp_img)
    if hand_res.hand_landmarks and hand_res.handedness:
        for hand_lms, handedness in zip(hand_res.hand_landmarks, hand_res.handedness):
            coords = np.array([[lm.x, lm.y, lm.z] for lm in hand_lms], dtype=np.float32).flatten()
            if handedness[0].category_name == 'Left':
                lh, has_lh = coords, True
            else:
                rh, has_rh = coords, True

    vec = np.empty(1530, dtype=np.float32)
    vec[:1404] = face
    vec[1404:1467] = lh
    vec[1467:] = rh
    return vec, has_lh or has_rh

print('[Cell 5a] extract_tier3_keypoints() ready via Tasks API (face=1404, lh=63, rh=63 -> 1530)')


In [ ]:
def to_fixed_sequence(seq, seq_len=30, feat_dim=1530):
    n = len(seq)
    if n == 0:
        return np.zeros((seq_len, feat_dim), dtype=np.float32)
    arr = np.asarray(seq, dtype=np.float32).reshape(n, feat_dim)
    if n >= seq_len:
        idx = np.linspace(0, n - 1, seq_len, dtype=np.int32)
        return arr[idx]
    out = np.zeros((seq_len, feat_dim), dtype=np.float32)
    out[:n] = arr
    return out
print('[Cell 5b/2] to_fixed_sequence() defined')


## Cell 6: Build Sample List


In [ ]:
samples = []
dataset_type = C['dataset_type']

if dataset_type == 'wlasl':
    wlasl_json = C['wlasl_json']
    videos_dir = C['videos_dir']

    print(f'Reading WLASL JSON: {wlasl_json}')
    with open(wlasl_json, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # ── Load NSLT split file ──────────────────────────────────────────────
    # nslt structure: {"video_id_str": {"subset": "train", "action": [class_idx, ...]}}
    # Keys may be zero-padded (e.g. '05237'); use int() to normalise both sides.
    nslt_path = C.get('nslt_json')
    nslt_vid_to_class = {}   # int(video_id) -> class_idx (= wlasl array index)
    if nslt_path and Path(nslt_path).exists():
        with open(nslt_path, 'r', encoding='utf-8') as f:
            nslt_data = json.load(f)
        for _k, _v in nslt_data.items():
            try:
                nslt_vid_to_class[int(_k)] = int(_v['action'][0])
            except (KeyError, ValueError, TypeError):
                pass
        nslt_class_ids = set(nslt_vid_to_class.values())
        print(f'NSLT filter loaded: {len(nslt_vid_to_class)} video IDs, {len(nslt_class_ids)} classes from {Path(nslt_path).name}')
    else:
        print('⚠️ No NSLT file found — using all videos')

    for idx, item in enumerate(data):
        # WLASL_v0.3.json has no gloss_id/class_id field; array index IS the class index
        wlasl_class_id = idx
        if wlasl_class_id not in allowed_class_ids:
            continue

        for inst in item.get('instances', []):
            vid = inst.get('video_id', inst.get('id', None))
            if vid is None:
                continue
            vid_int = int(vid)

            # Filter by NSLT; use class_id from nslt (authoritative benchmark label)
            if nslt_vid_to_class:
                class_id = nslt_vid_to_class.get(vid_int)
                if class_id is None:
                    continue
            else:
                class_id = wlasl_class_id

            if class_id not in classid_to_label:
                continue

            vp = videos_dir / f'{vid_int}.mp4'
            samples.append({
                'video_path' : vp,
                'class_id'   : class_id,
                'label_name' : classid_to_label[class_id],
            })

elif dataset_type == 'folder_classid':
    videos_dir = C['videos_dir']
    for class_dir in videos_dir.iterdir():
        if not class_dir.is_dir(): continue
        try: class_id = int(class_dir.name)
        except (ValueError, TypeError): continue
        if class_id not in allowed_class_ids: continue
        for vp in class_dir.rglob('*.mp4'):
            samples.append({'video_path': vp, 'class_id': class_id, 'label_name': classid_to_label[class_id]})

# Keep only existing videos
samples = [s for s in samples if s['video_path'].exists()]
print(f'✅ Indexed samples (files exist): {len(samples)}')



## Cell 7: Extract or Load Cache

_Four sub-cells: cache flag → load cache → extract loop → guard check_


In [ ]:
USE_CACHE_IF_EXISTS = False
cache_available = USE_CACHE_IF_EXISTS and CACHE_NPZ.exists()

print(f"[Cell 7a/4] Cache flag | USE_CACHE_IF_EXISTS={USE_CACHE_IF_EXISTS} | cache_available={cache_available} | path={CACHE_NPZ}")


In [ ]:
if cache_available:
    z = np.load(CACHE_NPZ, allow_pickle=True)
    X = z["X"]
    y_text = z["y_text"] if "y_text" in z else z["y"]
    print(f"✅ Loaded cache: {CACHE_NPZ}")
    print("X:", X.shape, "| y:", y_text.shape)
else:
    print("[Cell 7b/4] Cache not loaded (will extract from scratch)")


In [ ]:
if not cache_available:
    X_list, y_list = [], []
    skipped_missing = 0
    skipped_no_hand = 0

    for s in tqdm(samples, desc=f'Extracting ({LANGUAGE})'):
        vp = s['video_path']
        if not vp.exists():
            skipped_missing += 1
            continue

        cap = cv2.VideoCapture(str(vp))
        cap.set(cv2.CAP_PROP_BUFFERSIZE, 1)
        seq             = []
        total_frames    = 0
        detected_frames = 0
        frame_idx       = 0

        while True:
            if not cap.grab():
                break
            frame_idx += 1
            if frame_idx % FRAME_SKIP != 0:
                continue
            ok, frame = cap.retrieve()
            if not ok:
                continue
            total_frames += 1

            if RESIZE_WIDTH > 0:
                h, w = frame.shape[:2]
                new_h = int(h * RESIZE_WIDTH / w)
                frame = cv2.resize(frame, (RESIZE_WIDTH, new_h))

            vec, has_hand = extract_tier3_keypoints(frame)
            if has_hand:
                detected_frames += 1
            seq.append(vec)

        cap.release()

        if total_frames == 0:
            continue

        if (detected_frames / total_frames) < 0.2:
            skipped_no_hand += 1
            continue

        seq_fixed = to_fixed_sequence(seq, SEQUENCE_LENGTH, FEATURES_PER_FRAME)
        X_list.append(seq_fixed)
        y_list.append(s['label_name'])

    X = np.stack(X_list, axis=0)
    y_text = np.asarray(y_list, dtype=object)
    np.savez_compressed(CACHE_NPZ, X=X, y_text=y_text)
    print(f'Saved cache: {CACHE_NPZ}')
    print(f'X: {X.shape} | y: {y_text.shape}')
    print(f'Skipped (missing file): {skipped_missing} | Skipped (no hand): {skipped_no_hand}')



In [ ]:
if len(X) == 0:
    raise RuntimeError("No extracted samples. Check paths/vocab/dataset format.")

print(f"[Cell 7d/4] Extraction OK | total samples: {len(X)} | X.shape={X.shape}")


## Cell 8: Preprocess + Split


In [ ]:
N, T, F = X.shape
X2 = X.reshape(-1, F)

scaler = StandardScaler()
X2 = scaler.fit_transform(X2)
X_scaled = X2.reshape(N, T, F).astype(np.float32)

np.savez_compressed(
    SCALER_STATS,
    mean=scaler.mean_.astype(np.float32),
    scale=scaler.scale_.astype(np.float32),
)

le = LabelEncoder()
y_idx = le.fit_transform(y_text)
y_onehot = to_categorical(y_idx)

X_train, X_tmp, y_train, y_tmp = train_test_split(
    X_scaled, y_onehot, test_size=0.4, random_state=42, stratify=y_idx
)
y_tmp_idx = np.argmax(y_tmp, axis=1)
_, strat_cnt = np.unique(y_tmp_idx, return_counts=True)
_strat = y_tmp_idx if strat_cnt.min() >= 2 else None
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, random_state=42, stratify=_strat
)

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

# ── Dataset health metrics ────────────────────────────────────────────────
_counts = Counter(y_text)
_vals   = np.array(list(_counts.values()))
print(f'\nDataset overview:')
print(f'  Total samples : {N}')
print(f'  Classes       : {len(_counts)}')
print(f'  Samples/class : min={_vals.min()}  max={_vals.max()}  mean={_vals.mean():.1f}  median={int(np.median(_vals))}')
_rare = sum(1 for v in _vals if v < 5)
print(f'  Classes with <5 samples: {_rare} ({100*_rare/len(_counts):.0f}%)')
print(f'  Feature range after scaling: mean≈{X_scaled.mean():.3f}  std≈{X_scaled.std():.3f}')


In [ ]:
# Save per-language class mapping
classes_df = pd.DataFrame(
    {"model_class_index": np.arange(len(le.classes_)), "label_name": le.classes_}
)
label_to_src = dict(zip(vocab["label_name"], vocab["source_class_id"]))
classes_df["source_class_id"] = classes_df["label_name"].map(label_to_src)

# Keep word_id if present (makes output compatible with the existing Live Test notebooks/scripts)
if "word_id" in vocab.columns:
    label_to_word = dict(zip(vocab["label_name"], vocab["word_id"]))
    classes_df["word_id"] = classes_df["label_name"].map(label_to_word)

classes_df.to_csv(CLASSES_CSV, index=False)
print("✅ Saved classes:", CLASSES_CSV)
display(classes_df.head())


## Cell 8b: Data Augmentation

With ~10 training samples per class, augmentation is the single biggest lever.
Four techniques are applied to the **training set only** (val/test stay clean):
- **Gaussian keypoint noise** – random jitter on landmark positions
- **Temporal shift** – circularly roll the sequence ±3 frames
- **Frame masking** – randomly zero ~10 % of frames
- **Speed perturbation** – resample the sequence at 75 %–125 % of its original speed

`AUG_FACTOR` copies are generated → training set is multiplied by `AUG_FACTOR + 1`.


In [ ]:
AUG_FACTOR = 4  # training set will grow to (AUG_FACTOR+1) × original size

def _speed_perturb(seq, min_rate=0.75, max_rate=1.25):
    T, F = seq.shape
    rate  = np.random.uniform(min_rate, max_rate)
    n_new = max(1, int(T * rate))
    idx   = np.linspace(0, T - 1, n_new).astype(np.int32)
    stretched = seq[idx]
    if n_new >= T:
        return stretched[:T]
    pad = np.zeros((T - n_new, F), dtype=seq.dtype)
    return np.concatenate([stretched, pad], axis=0)

def augment_sequence(seq, noise_std=0.03, max_shift=3, mask_prob=0.10):
    """Apply random augmentations to a (T, F) float32 sequence."""
    aug = seq.copy()
    aug += np.random.normal(0, noise_std, aug.shape).astype(np.float32)
    aug  = np.roll(aug, np.random.randint(-max_shift, max_shift + 1), axis=0)
    mask = np.random.random(aug.shape[0]) < mask_prob
    aug[mask] = 0.0
    aug  = _speed_perturb(aug).astype(np.float32)
    return aug

print(f'Augmenting training set {AUG_FACTOR}x ...')
_aug_X = [X_train]
_aug_y = [y_train]
for _ in range(AUG_FACTOR):
    _aug_X.append(np.stack([augment_sequence(s) for s in X_train]))
    _aug_y.append(y_train)

X_train_aug = np.concatenate(_aug_X, axis=0)
y_train_aug = np.concatenate(_aug_y, axis=0)

_perm       = np.random.permutation(len(X_train_aug))
X_train_aug = X_train_aug[_perm]
y_train_aug = y_train_aug[_perm]

print(f'  Original train : {X_train.shape}')
print(f'  Augmented train: {X_train_aug.shape}')
print(f'  Val / Test unchanged: {X_val.shape}  /  {X_test.shape}')


## Cell 9: Build Model


In [ ]:
num_classes = y_train.shape[1]

# ── Input projection: 1530 → 512 per frame (reduces LSTM workload) ───────
inp = Input(shape=(SEQUENCE_LENGTH, FEATURES_PER_FRAME), name='sequence_input')
x   = TimeDistributed(Dense(512, use_bias=False), name='proj')(inp)
x   = BatchNormalization()(x)
x   = Activation('relu')(x)
x   = Dropout(0.3)(x)

# ── BiLSTM stack ──────────────────────────────────────────────────────────
x = Bidirectional(LSTM(256, return_sequences=True), name='bilstm_1')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

x = Bidirectional(LSTM(128, return_sequences=True), name='bilstm_2')(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)

# ── Temporal Attention ────────────────────────────────────────────────────
# Learn which frames matter most for the prediction.
attn = Dense(1, activation='tanh', name='attn_score')(x)          # (B, T, 1)
attn = tf.keras.layers.Softmax(axis=1, name='attn_weight')(attn)  # (B, T, 1)
x    = Multiply(name='attn_apply')([x, attn])                     # weighted frames
x    = Lambda(lambda z: tf.reduce_sum(z, axis=1), name='attn_pool')(x)  # (B, 256)

# ── Classification head ───────────────────────────────────────────────────
x   = Dense(256, activation='relu', name='fc1')(x)
x   = BatchNormalization()(x)
x   = Dropout(0.35)(x)
out = Dense(num_classes, activation='softmax', dtype='float32', name='output')(x)

model = Model(inp, out, name='asl_bilstm_attention')
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
    metrics=[
        'accuracy',
        tf.keras.metrics.TopKCategoricalAccuracy(k=5, name='top5_acc'),
    ],
)
model.summary()


## Cell 10: Train


In [ ]:
# ── Class weights (computed on original labels, not augmented) ───────────
y_train_idx  = np.argmax(y_train, axis=1)
_classes     = np.unique(y_train_idx)
_weights     = compute_class_weight('balanced', classes=_classes, y=y_train_idx)
class_weight = dict(zip(_classes.tolist(), _weights.tolist()))
print(f'Class weight range: min={min(_weights):.2f}  max={max(_weights):.2f}')

# ── LR schedule: linear warmup → cosine decay ─────────────────────────────
_EPOCHS    = 200
_bs        = max(16, min(64, len(X_train_aug) // 40))
_WARMUP    = 10     # epochs of linear warmup
_LR_PEAK   = 1e-3
_LR_MIN    = 1e-6
_steps_ep  = max(1, len(X_train_aug) // _bs)

class _WarmupCosineDecay(tf.keras.callbacks.Callback):
    """Linear warmup for _WARMUP epochs then cosine decay to _LR_MIN."""
    def on_epoch_begin(self, epoch, logs=None):
        if epoch < _WARMUP:
            lr = _LR_PEAK * (epoch + 1) / _WARMUP
        else:
            import math
            prog = (epoch - _WARMUP) / max(1, _EPOCHS - _WARMUP)
            lr   = _LR_MIN + 0.5 * (_LR_PEAK - _LR_MIN) * (1 + math.cos(math.pi * prog))
        tf.keras.backend.set_value(self.model.optimizer.learning_rate, float(lr))

callbacks = [
    ModelCheckpoint(
        str(MODEL_BEST),
        monitor='val_accuracy',
        save_best_only=True,
        mode='max',
        verbose=1,
    ),
    EarlyStopping(
        monitor='val_accuracy', patience=30, restore_best_weights=True, verbose=1
    ),
    _WarmupCosineDecay(),
]

print(f'Batch size: {_bs}  |  Augmented training samples: {len(X_train_aug)}')
print(f'Epochs: {_EPOCHS}  |  Warmup: {_WARMUP}  |  LR peak: {_LR_PEAK}  |  LR min: {_LR_MIN}')

history = model.fit(
    X_train_aug,
    y_train_aug,
    validation_data=(X_val, y_val),
    epochs=_EPOCHS,
    batch_size=_bs,
    class_weight=class_weight,
    callbacks=callbacks,
    verbose=1,
)

model.save(MODEL_FINAL)
print('✅ Saved best :', MODEL_BEST)
print('✅ Saved final:', MODEL_FINAL)

# ── Training curve plots ──────────────────────────────────────────────────
import matplotlib.pyplot as plt

_h   = history.history
_eps = range(1, len(_h['accuracy']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

# Accuracy
axes[0].plot(_eps, _h['accuracy'],     label='Train acc')
axes[0].plot(_eps, _h['val_accuracy'], label='Val acc')
axes[0].set_title('Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()
axes[0].grid(True)

# Loss
axes[1].plot(_eps, _h['loss'],     label='Train loss')
axes[1].plot(_eps, _h['val_loss'], label='Val loss')
axes[1].set_title('Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()
axes[1].grid(True)

# Top-5 accuracy
axes[2].plot(_eps, _h['top5_acc'],     label='Train top-5')
axes[2].plot(_eps, _h['val_top5_acc'], label='Val top-5')
axes[2].set_title('Top-5 Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].legend()
axes[2].grid(True)

plt.suptitle(f'Training History  ({LANGUAGE.upper()}, {num_classes} classes)', fontsize=13)
plt.tight_layout()
plt.savefig(str(WORK_DIR / f'{LANGUAGE}_training_curves.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'Best val_accuracy : {max(_h["val_accuracy"]):.4f}  (epoch {np.argmax(_h["val_accuracy"])+1})')
print(f'Best val_top5_acc : {max(_h["val_top5_acc"]):.4f}')
print(f'Epochs run        : {len(_eps)}')


## Cell 11: Evaluation


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

_pbs = max(1, min(256, len(X_test)))

# ── Core metrics ──────────────────────────────────────────────────────────
loss, acc, top5 = model.evaluate(X_test, y_test, verbose=0, batch_size=_pbs)
y_prob = model.predict(X_test, verbose=0, batch_size=_pbs)
y_true = np.argmax(y_test, axis=1)
y_pred = np.argmax(y_prob, axis=1)

# Top-k accuracy helper
def topk_acc(y_true_idx, y_prob, k):
    topk = np.argsort(y_prob, axis=1)[:, -k:]
    return np.mean([y_true_idx[i] in topk[i] for i in range(len(y_true_idx))])

top1  = np.mean(y_true == y_pred)
top3  = topk_acc(y_true, y_prob, 3)
top5v = topk_acc(y_true, y_prob, 5)

print('=' * 50)
print(f'  Test Loss   : {loss:.4f}')
print(f'  Top-1 Acc   : {top1:.4f}  ({top1*100:.1f}%)')
print(f'  Top-3 Acc   : {top3:.4f}  ({top3*100:.1f}%)')
print(f'  Top-5 Acc   : {top5v:.4f}  ({top5v*100:.1f}%)')
print('=' * 50)

# ── Per-class summary ─────────────────────────────────────────────────────
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score, f1_score

bal_acc   = balanced_accuracy_score(y_true, y_pred)
macro_f1  = f1_score(y_true, y_pred, average='macro',    zero_division=0)
weighted_f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)
print(f'  Balanced Acc: {bal_acc:.4f}')
print(f'  Macro F1    : {macro_f1:.4f}')
print(f'  Weighted F1 : {weighted_f1:.4f}')
print()

print('Classification report:')
print(classification_report(y_true, y_pred,
                             labels=np.arange(len(le.classes_)),
                             target_names=le.classes_,
                             zero_division=0))

# ── Best / worst classes ──────────────────────────────────────────────────
from sklearn.metrics import classification_report as _cr
_rep = _cr(y_true, y_pred, labels=np.arange(len(le.classes_)),
           target_names=le.classes_, zero_division=0, output_dict=True)
_class_f1 = {k: v['f1-score'] for k, v in _rep.items()
             if k not in ('accuracy','macro avg','weighted avg')}
_sorted   = sorted(_class_f1.items(), key=lambda x: x[1], reverse=True)
print('Top-10 best classes (F1):')
for _name, _f1 in _sorted[:10]:
    print(f'  {_name:<20s}  F1={_f1:.3f}')
print('Top-10 worst classes (F1):')
for _name, _f1 in _sorted[-10:]:
    print(f'  {_name:<20s}  F1={_f1:.3f}')

# ── Confusion matrix plot (top-N most confused classes) ───────────────────
_cm      = confusion_matrix(y_true, y_pred)
_cm_norm = _cm.astype(float) / (_cm.sum(axis=1, keepdims=True) + 1e-9)

# Select classes that appear in test set
_active_idx = np.where(_cm.sum(axis=1) > 0)[0]
_n_show     = min(40, len(_active_idx))
_top_idx    = _active_idx[np.argsort(_cm_norm.diagonal()[_active_idx])[::-1][:_n_show]]
_cm_show    = _cm_norm[np.ix_(_top_idx, _top_idx)]
_labels_show = le.classes_[_top_idx]

fig, ax = plt.subplots(figsize=(max(10, _n_show // 2), max(8, _n_show // 2)))
im = ax.imshow(_cm_show, interpolation='nearest', cmap='Blues', vmin=0, vmax=1)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
ax.set_xticks(range(_n_show))
ax.set_yticks(range(_n_show))
ax.set_xticklabels(_labels_show, rotation=90, fontsize=7)
ax.set_yticklabels(_labels_show, fontsize=7)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title(f'Normalised Confusion Matrix (top {_n_show} classes by recall)')
plt.tight_layout()
plt.savefig(str(WORK_DIR / f'{LANGUAGE}_confusion_matrix.png'), dpi=120, bbox_inches='tight')
plt.show()

# ── Confidence histogram ──────────────────────────────────────────────────
_conf_correct = y_prob[np.arange(len(y_true)), y_pred][y_true == y_pred]
_conf_wrong   = y_prob[np.arange(len(y_true)), y_pred][y_true != y_pred]
fig2, ax2 = plt.subplots(figsize=(8, 4))
ax2.hist(_conf_correct, bins=20, alpha=0.7, label=f'Correct ({len(_conf_correct)})', color='steelblue')
ax2.hist(_conf_wrong,   bins=20, alpha=0.7, label=f'Wrong   ({len(_conf_wrong)})',   color='tomato')
ax2.set_xlabel('Predicted confidence (softmax max)')
ax2.set_ylabel('Count')
ax2.set_title('Prediction Confidence Distribution')
ax2.legend()
ax2.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(str(WORK_DIR / f'{LANGUAGE}_confidence_hist.png'), dpi=120, bbox_inches='tight')
plt.show()
print(f'Avg confidence (correct): {_conf_correct.mean():.3f}' if len(_conf_correct) else '')
print(f'Avg confidence (wrong)  : {_conf_wrong.mean():.3f}'   if len(_conf_wrong)   else '')


## Cell 12: Summary


In [ ]:
print('=' * 60)
print('✅  TRAINING COMPLETE')
print('=' * 60)
print(f'  Language     : {LANGUAGE.upper()}')
print(f'  Classes      : {num_classes}')
print(f'  Samples      : train={len(X_train)}  val={len(X_val)}  test={len(X_test)}')
print(f'  Feature dim  : {FEATURES_PER_FRAME}  |  Seq length: {SEQUENCE_LENGTH}')
print()
print('Model results:')
print(f'  Top-1 test acc   : {top1*100:.2f}%')
print(f'  Top-3 test acc   : {top3*100:.2f}%')
print(f'  Top-5 test acc   : {top5v*100:.2f}%')
print(f'  Balanced acc     : {bal_acc*100:.2f}%')
print(f'  Macro F1         : {macro_f1:.4f}')
print(f'  Weighted F1      : {weighted_f1:.4f}')
print(f'  Best val acc     : {max(history.history["val_accuracy"])*100:.2f}%')
print()
print('Output files:')
print(f'  Vocab CSV        : {VOCAB_CSV}')
print(f'  Classes CSV      : {CLASSES_CSV}')
print(f'  Cache NPZ        : {CACHE_NPZ}')
print(f'  Scaler stats     : {SCALER_STATS}')
print(f'  Model (best)     : {MODEL_BEST}')
print(f'  Model (final)    : {MODEL_FINAL}')
print(f'  Training curves  : {WORK_DIR / (LANGUAGE + "_training_curves.png")}')
print(f'  Confusion matrix : {WORK_DIR / (LANGUAGE + "_confusion_matrix.png")}')
print(f'  Confidence hist  : {WORK_DIR / (LANGUAGE + "_confidence_hist.png")}')
print('=' * 60)
